In [189]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
import random
%matplotlib inline

In [190]:
words = open('../names.txt').read().splitlines()

In [191]:
stoi = {c:i+1 for i,c in enumerate(sorted(set(''.join(words))))}
stoi['.'] = 0
itos = {i:c for c, i in stoi.items()}

In [192]:
def build_dataset(data):
    block_size = 3
    X = []
    Y = []
    for word in data:
        start = [0] * (block_size-1)
        paddedWord = '.' + word + '.'
        subL = []
        for c in paddedWord:
            subL.append(stoi[c])
        for x, y in zip(subL, subL[1:]):
            start.append(x)
            X.append(start)
            Y.append(y)
            start = start[1:]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X,Y


In [193]:
random.seed(42)
random.shuffle(words)
eightyP = int(.8 * len(words))
ninetyP = int(.9 * len(words))
Xtrain, Ytrain = build_dataset(words[:eightyP])
Xdev, Ydev = build_dataset(words[eightyP:ninetyP])
Xtest, Ytest = build_dataset(words[ninetyP:])

In [194]:
g = torch.Generator().manual_seed(42)
C = torch.randn(27, 2, generator=g, requires_grad=True)
W1 = torch.randn(6, 100, generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn(100, 27, generator=g, requires_grad=True)
b2 = torch.randn(27, generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]

In [195]:
def zeroGrad():
    for val in parameters:
        val.grad = None

In [196]:
def weightUpdate():
    for val in parameters:
        val.data -= 0.1 * val.grad

In [197]:
def trainingLoop():
    for _ in range(100):
        emb = torch.flatten(C[Xtrain], start_dim=1)
        h = torch.tanh(emb @ W1 + b1)
        logits = h @ W2 + b2
        #counts = torch.exp(logits)
        #probs = counts/counts.sum(dim=1, keepdim=True)
        #nll = torch.log(probs[torch.arange(32), Yt])
        #loss = -torch.mean(nll)

        loss = F.cross_entropy(logits, Ytrain)
        print(loss.item())
        zeroGrad()
        loss.backward()
        weightUpdate()

        

In [198]:
trainingLoop()

16.72115135192871
14.935977935791016
13.85555648803711
12.996589660644531
12.285560607910156
11.726740837097168
11.265556335449219
10.855612754821777
10.47636890411377
10.133463859558105
9.827049255371094
9.54954719543457
9.294520378112793
9.058237075805664
8.838427543640137
8.633238792419434
8.440717697143555
8.258936882019043
8.08629035949707
7.921563148498535
7.763874530792236
7.612590789794922
7.467238426208496
7.327449798583984
7.192923545837402
7.063398838043213
6.9386420249938965
6.818426609039307
6.702537536621094
6.590770721435547
6.482940673828125
6.378878116607666
6.278445720672607
6.18152379989624
6.088014602661133
5.99783182144165
5.91089391708374
5.827110767364502
5.746379852294922
5.6685872077941895
5.59360408782959
5.521297931671143
5.451534748077393
5.384188175201416
5.319141387939453
5.256285190582275
5.195522308349609
5.136758327484131
5.0799055099487305
5.024880409240723
4.971599578857422
4.919987678527832
4.869969367980957
4.8214802742004395
4.774456977844238
4.728